In [1]:
from Bio import SeqIO, Phylo
import numpy as np
import pandas as pd
from tqdm import tqdm
import sys
sys.path.append('../pysimARG')
from fasta_to_bool import fasta_to_bool
from segment_summary_stats import segment_summary_stats
from clonal_genealogy import ClonalTree

## Genome data

In [2]:
genome_names = []
for record in SeqIO.parse("../data/SimBac/genomes.fasta", "fasta"):
    print(record.id, len(record.seq))
    genome_names.append(record.id)

0 1000000
1 1000000
2 1000000
3 1000000
4 1000000
5 1000000
6 1000000
7 1000000
8 1000000
9 1000000
10 1000000
11 1000000
12 1000000
13 1000000
14 1000000
15 1000000
16 1000000
17 1000000
18 1000000
19 1000000
20 1000000
21 1000000
22 1000000
23 1000000
24 1000000
25 1000000
26 1000000
27 1000000
28 1000000
29 1000000


## Load `.nwk` tree

In [4]:
clonal_tree = Phylo.read("../data/SimBac/final_clonal.nwk", "newick")
Phylo.draw_ascii(clonal_tree)

                                                                 _______ 17
  ______________________________________________________________|
 |                                                              |     __ 19
 |                                                              |____|
 |                                                                   |__ 9
 |
 |                                                                  ____ 14
 |                                                                 |
_|                                                                 |    , 15
 |                                                      ___________|   _|
 |                                                     |           |  | , 5
 |                                                     |           |  | |
 |                                                     |           |__| | 24
 |                                                     |              |
 |                                    

In [5]:
clonal_edge = np.loadtxt("../data/SimBac/clonal_edge.csv", delimiter=",", dtype=float)
clonal_node_height = np.loadtxt("../data/SimBac/clonal_node_height.csv", delimiter=",", dtype=float)

In [6]:
clonal_edge, clonal_node_height

(array([[3.10000000e+01, 3.00000000e+00, 6.69683962e-03],
        [3.10000000e+01, 4.00000000e+00, 6.69683962e-03],
        [3.20000000e+01, 2.00000000e+00, 8.53338822e-03],
        [3.20000000e+01, 3.10000000e+01, 1.83654860e-03],
        [3.30000000e+01, 5.00000000e+00, 8.83634098e-03],
        [3.30000000e+01, 6.00000000e+00, 8.83634098e-03],
        [3.40000000e+01, 1.20000000e+01, 1.55289811e-02],
        [3.40000000e+01, 1.30000000e+01, 1.55289811e-02],
        [3.50000000e+01, 1.60000000e+01, 2.08754379e-02],
        [3.50000000e+01, 1.70000000e+01, 2.08754379e-02],
        [3.60000000e+01, 1.80000000e+01, 2.36852587e-02],
        [3.60000000e+01, 1.90000000e+01, 2.36852587e-02],
        [3.70000000e+01, 2.20000000e+01, 2.61890864e-02],
        [3.70000000e+01, 2.30000000e+01, 2.61890864e-02],
        [3.80000000e+01, 2.00000000e+01, 3.54862213e-02],
        [3.80000000e+01, 3.60000000e+01, 1.18009627e-02],
        [3.90000000e+01, 2.40000000e+01, 3.74600840e-02],
        [3.900

## Load genome data and transfer to boolean matrix

In [7]:
leaf_names = np.loadtxt("../data/SimBac/tip_names.csv", delimiter=",", dtype=str)
leaf_names = np.char.strip(leaf_names, '"')
len(leaf_names), len(genome_names)

(30, 30)

In [8]:
bool_mat = fasta_to_bool("../data/SimBac/genomes.fasta")
bool_mat.shape, bool_mat.dtype

((30, 1000000), dtype('bool'))

In [9]:
index_map = {string_id: idx for idx, string_id in enumerate(genome_names)}
new_indices = [index_map[string_id] for string_id in leaf_names]
genomes_bool = bool_mat[new_indices]
genomes_bool.shape, genomes_bool.dtype

((30, 1000000), dtype('bool'))

In [10]:
np.savetxt("../data/SimBac/genomes_bool.csv", genomes_bool, delimiter=",", fmt="%d")
# bool_mat = np.loadtxt("../data/SimBac/genomes_bool.csv", delimiter=",", dtype=bool)
# bool_mat.shape, bool_mat.dtype

## Compute estimated theta

In [11]:
has_true = genomes_bool.any(axis=0)
has_false = ~genomes_bool.all(axis=0)
idx_seg = np.where(has_true & has_false)[0]

In [12]:
count_S = idx_seg.size
count_S

161451

In [13]:
genomes_bool.shape[1]

1000000

In [14]:
np.sum(clonal_edge[:, 2])

np.float64(6.901593076493213)

theta / 2 = S / L / tree length

In [15]:
print("theta:", float(count_S / genomes_bool.shape[1] / np.sum(clonal_edge[:, 2]) * 2))

theta: 0.046786589186169555


## Get summary statistics for random genome segments with random length

Randomly select 2000 segments with windows of between 100 to 10000 bp each.

In [16]:
rand_seg_df = pd.DataFrame({'Gene_Length': np.hstack([np.repeat(None, 2000)])})
rand_seg_df['Start_pos'] = None
rand_seg_df['End_pos'] = None
rand_seg_df['Alignment'] = True
rand_seg_df

,Gene_Length,Start_pos,End_pos,Alignment
0,None,None,None,True
1,None,None,None,True
2,None,None,None,True
3,None,None,None,True
4,None,None,None,True
...,...,...,...,...
1995,None,None,None,True
1996,None,None,None,True
1997,None,None,None,True
1998,None,None,None,True


In [17]:
np.random.seed(100)

max_start_pos = genomes_bool.shape[1] - 10000 + 1
for i in tqdm(range(2000), desc="Processing genome segments"):
    start_pos = np.random.randint(1, max_start_pos + 1)
    length = np.random.randint(100, 10001)  # Random length between 100 and 10000
    end_pos = start_pos + length - 1
    rand_seg_df.loc[i, 'Gene_Length'] = length
    rand_seg_df.loc[i, 'Start_pos'] = start_pos
    rand_seg_df.loc[i, 'End_pos'] = end_pos

print(rand_seg_df.head())
rand_seg_df.to_csv("../data/SimBac/rand_seg_info.csv", index=False)

Processing genome segments: 100%|██████████| 2000/2000 [00:00<00:00, 6596.91it/s]

  Gene_Length Start_pos End_pos  Alignment
0        7036    824841  831876       True
1        8139    210756  218894       True
2         179    864088  864266       True
3         450    212913  213362       True
4        4249    931765  936013       True
